In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from os import environ
import re

/opt/anaconda3/envs/venv_personal/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Splitting documents

In [2]:
# Load documents
documents_1 = ''

with open("../data/activities.txt", "r") as file:
    documents_1 = file.read()

In [3]:
print(documents_1[:500])

Activities in Jakarta, Indonesia.

# Activity: Taman Mini Indonesia Indah – Jakarta Cultural Theme Park (Fee: Rp 31,000).

Taman Mini Indonesia Indah is a cultural theme park in East Jakarta that highlights the traditions and heritage of Indonesia’s provinces through traditional pavilions, clothing, dances, and regional customs. The park features a central lake with a miniature of the Indonesian archipelago, a cable car, museums, the IMAX Theater, and Tanah Airku Theatre, offering visitors a com


In [3]:
# Remove the \n and retain \n \n
documents_1 = documents_1.replace("\n \n", "new_paragraph")
documents_1 = documents_1.replace("\n", " ")
documents_1 = documents_1.replace("new_paragraph", "\n \n")

In [4]:
# Document Splitting
chunk_size = 250
chunk_overlap = 50

splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separators=["#", "\n\n", "\n", " "]
)
split_1 = splitter.split_text(documents_1)
split_1 = splitter.create_documents(split_1)

In [6]:
split_1[:5]

[Document(metadata={}, page_content='Activities in Jakarta, Indonesia.'),
 Document(metadata={}, page_content='# Activity: Taman Mini Indonesia Indah – Jakarta Cultural Theme Park (Fee: Rp 31,000).  Taman Mini Indonesia Indah is a cultural theme park in East Jakarta that highlights the traditions and heritage of Indonesia’s provinces through traditional'),
 Document(metadata={}, page_content='of Indonesia’s provinces through traditional pavilions, clothing, dances, and regional customs. The park features a central lake with a miniature of the Indonesian archipelago, a cable car, museums, the IMAX Theater, and Tanah Airku Theatre,'),
 Document(metadata={}, page_content='the IMAX Theater, and Tanah Airku Theatre, offering visitors a comprehensive look at Indonesia’s cultural diversity and recreational attractions.  The park showcases the daily life and traditions of 26 provinces with authentic regional architecture.'),
 Document(metadata={}, page_content='provinces with authentic regiona

In [5]:
text_chunks = []
subtitle = ""
for chunk in split_1:
    if "#" in chunk.page_content:
        match = re.search(r"#.*?(?=\s{2}|$)", chunk.page_content)
        if match:
            subtitle = match.group()

    doc = Document(
        page_content=f"({subtitle}) {chunk.page_content}",
        metadata={
            "subtitle": subtitle,
            "content": chunk.page_content
        }
    )
    text_chunks.append(doc)

In [8]:
text_chunks[:5]

[Document(metadata={'subtitle': '', 'content': 'Activities in Jakarta, Indonesia.'}, page_content='() Activities in Jakarta, Indonesia.'),
 Document(metadata={'subtitle': '# Activity: Taman Mini Indonesia Indah – Jakarta Cultural Theme Park (Fee: Rp 31,000).', 'content': '# Activity: Taman Mini Indonesia Indah – Jakarta Cultural Theme Park (Fee: Rp 31,000).  Taman Mini Indonesia Indah is a cultural theme park in East Jakarta that highlights the traditions and heritage of Indonesia’s provinces through traditional'}, page_content='(# Activity: Taman Mini Indonesia Indah – Jakarta Cultural Theme Park (Fee: Rp 31,000).) # Activity: Taman Mini Indonesia Indah – Jakarta Cultural Theme Park (Fee: Rp 31,000).  Taman Mini Indonesia Indah is a cultural theme park in East Jakarta that highlights the traditions and heritage of Indonesia’s provinces through traditional'),
 Document(metadata={'subtitle': '# Activity: Taman Mini Indonesia Indah – Jakarta Cultural Theme Park (Fee: Rp 31,000).', 'conte

In [9]:
text_chunks[-5:]

[Document(metadata={'subtitle': '# Activity: Playtopia Sports Gokart Ticket – Lippo Mall Puri Racing Experience (Rp 150,000).', 'content': '# Activity: Playtopia Sports Gokart Ticket – Lippo Mall Puri Racing Experience (Rp 150,000).  Experience Indonesia’s first multi-level indoor karting track at Playtopia Sports Gokart Lippo Mall Puri, where the thrill of racing meets the excitement of'}, page_content='(# Activity: Playtopia Sports Gokart Ticket – Lippo Mall Puri Racing Experience (Rp 150,000).) # Activity: Playtopia Sports Gokart Ticket – Lippo Mall Puri Racing Experience (Rp 150,000).  Experience Indonesia’s first multi-level indoor karting track at Playtopia Sports Gokart Lippo Mall Puri, where the thrill of racing meets the excitement of'),
 Document(metadata={'subtitle': '# Activity: Playtopia Sports Gokart Ticket – Lippo Mall Puri Racing Experience (Rp 150,000).', 'content': 'the thrill of racing meets the excitement of video games. Race through spacious levels with advanced go

In [10]:
print(text_chunks[2].metadata["subtitle"])
print(text_chunks[2].metadata["content"])
print(text_chunks[2].page_content)

# Activity: Taman Mini Indonesia Indah – Jakarta Cultural Theme Park (Fee: Rp 31,000).
of Indonesia’s provinces through traditional pavilions, clothing, dances, and regional customs. The park features a central lake with a miniature of the Indonesian archipelago, a cable car, museums, the IMAX Theater, and Tanah Airku Theatre,
(# Activity: Taman Mini Indonesia Indah – Jakarta Cultural Theme Park (Fee: Rp 31,000).) of Indonesia’s provinces through traditional pavilions, clothing, dances, and regional customs. The park features a central lake with a miniature of the Indonesian archipelago, a cable car, museums, the IMAX Theater, and Tanah Airku Theatre,


In [11]:
print(text_chunks[0].metadata["content"])
print(text_chunks[1].metadata["content"])
print(text_chunks[2].metadata["content"])
print(text_chunks[3].metadata["content"])
print(text_chunks[4].metadata["content"])
print(text_chunks[5].metadata["content"])
print(text_chunks[6].metadata["content"])

Activities in Jakarta, Indonesia.
# Activity: Taman Mini Indonesia Indah – Jakarta Cultural Theme Park (Fee: Rp 31,000).  Taman Mini Indonesia Indah is a cultural theme park in East Jakarta that highlights the traditions and heritage of Indonesia’s provinces through traditional
of Indonesia’s provinces through traditional pavilions, clothing, dances, and regional customs. The park features a central lake with a miniature of the Indonesian archipelago, a cable car, museums, the IMAX Theater, and Tanah Airku Theatre,
the IMAX Theater, and Tanah Airku Theatre, offering visitors a comprehensive look at Indonesia’s cultural diversity and recreational attractions.  The park showcases the daily life and traditions of 26 provinces with authentic regional architecture.
provinces with authentic regional architecture. Recreational facilities include a cable car for aerial views and several museums for cultural exploration. The central lake features a detailed miniature representation of the entir

# Embedding and storing

In [6]:
# Load the saved embeddings model
embeddings = HuggingFaceEmbeddings(model_name="../saved_models/all-MiniLM-L6-v2")

In [7]:
# Implement embeddings
db = FAISS.from_documents(text_chunks, embeddings)

# Save db
db.save_local('../vector_store/activities')

# Similarity search

In [14]:
# Load db
load_db = FAISS.load_local(
    '../vector_store/activities', embeddings
)

In [15]:
# Retrieve answer
question = 'Is there a water related attraction?'

search = load_db.similarity_search(question)
search

[Document(metadata={'subtitle': '# Activity: Atlantis Water Adventure Ancol – Jakarta Water Park (Rp 112,000).', 'content': 'The park features adrenaline-pumping slides, relaxing pools, and attractions for all ages, making it ideal for family outings and group fun. Visitors can enjoy a day filled with excitement, from thrilling rides to dedicated kids’ areas.  The park'}, page_content='(# Activity: Atlantis Water Adventure Ancol – Jakarta Water Park (Rp 112,000).) The park features adrenaline-pumping slides, relaxing pools, and attractions for all ages, making it ideal for family outings and group fun. Visitors can enjoy a day filled with excitement, from thrilling rides to dedicated kids’ areas.  The park'),
 Document(metadata={'subtitle': '# Activity: Atlantis Water Adventure Ancol – Jakarta Water Park (Rp 112,000).', 'content': 'rides to dedicated kids’ areas.  The park offers popular attractions like Crazy Slide, Skybox, Poseidon Wave, Antila River, and Kiddy Pool. Locker rentals a

In [16]:
# Query more or less text chunks
search = load_db.similarity_search(question, k=6)
search

[Document(metadata={'subtitle': '# Activity: Atlantis Water Adventure Ancol – Jakarta Water Park (Rp 112,000).', 'content': 'The park features adrenaline-pumping slides, relaxing pools, and attractions for all ages, making it ideal for family outings and group fun. Visitors can enjoy a day filled with excitement, from thrilling rides to dedicated kids’ areas.  The park'}, page_content='(# Activity: Atlantis Water Adventure Ancol – Jakarta Water Park (Rp 112,000).) The park features adrenaline-pumping slides, relaxing pools, and attractions for all ages, making it ideal for family outings and group fun. Visitors can enjoy a day filled with excitement, from thrilling rides to dedicated kids’ areas.  The park'),
 Document(metadata={'subtitle': '# Activity: Atlantis Water Adventure Ancol – Jakarta Water Park (Rp 112,000).', 'content': 'rides to dedicated kids’ areas.  The park offers popular attractions like Crazy Slide, Skybox, Poseidon Wave, Antila River, and Kiddy Pool. Locker rentals a

In [17]:
search_scores = load_db.similarity_search_with_score(question)
search_scores

[(Document(metadata={'subtitle': '# Activity: Atlantis Water Adventure Ancol – Jakarta Water Park (Rp 112,000).', 'content': 'The park features adrenaline-pumping slides, relaxing pools, and attractions for all ages, making it ideal for family outings and group fun. Visitors can enjoy a day filled with excitement, from thrilling rides to dedicated kids’ areas.  The park'}, page_content='(# Activity: Atlantis Water Adventure Ancol – Jakarta Water Park (Rp 112,000).) The park features adrenaline-pumping slides, relaxing pools, and attractions for all ages, making it ideal for family outings and group fun. Visitors can enjoy a day filled with excitement, from thrilling rides to dedicated kids’ areas.  The park'),
  1.2194163),
 (Document(metadata={'subtitle': '# Activity: Atlantis Water Adventure Ancol – Jakarta Water Park (Rp 112,000).', 'content': 'rides to dedicated kids’ areas.  The park offers popular attractions like Crazy Slide, Skybox, Poseidon Wave, Antila River, and Kiddy Pool. 

In [18]:
load_db.similarity_search_with_score(question, k=8)

[(Document(metadata={'subtitle': '# Activity: Atlantis Water Adventure Ancol – Jakarta Water Park (Rp 112,000).', 'content': 'The park features adrenaline-pumping slides, relaxing pools, and attractions for all ages, making it ideal for family outings and group fun. Visitors can enjoy a day filled with excitement, from thrilling rides to dedicated kids’ areas.  The park'}, page_content='(# Activity: Atlantis Water Adventure Ancol – Jakarta Water Park (Rp 112,000).) The park features adrenaline-pumping slides, relaxing pools, and attractions for all ages, making it ideal for family outings and group fun. Visitors can enjoy a day filled with excitement, from thrilling rides to dedicated kids’ areas.  The park'),
  1.2194163),
 (Document(metadata={'subtitle': '# Activity: Atlantis Water Adventure Ancol – Jakarta Water Park (Rp 112,000).', 'content': 'rides to dedicated kids’ areas.  The park offers popular attractions like Crazy Slide, Skybox, Poseidon Wave, Antila River, and Kiddy Pool. 